# LeetCode 203 — Remove Linked List Elements bằng Python

**Link đề:** https://leetcode.com/problems/remove-linked-list-elements/
**Độ khó:** Easy · **Chủ đề:** Linked List, Recursion

```python
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def removeElements(self, head: ListNode | None, val: int) -> ListNode | None:
        ...
```

## Mục tiêu học
1. Hiểu **singly linked list**: node, `val`, `next`, `head`, `None`.
2. Biết "xóa" một node nghĩa là **nối vòng qua** nó: `prev.next = prev.next.next`.
3. Thấy vì sao xóa **node đầu** là trường hợp khó, và dùng **dummy node** (node giả) để xóa mọi trường hợp như nhau.
4. Viết 4 lời giải: tạo list mới → xử lý head riêng → dummy node → đệ quy; so sánh Time/Space.
5. Trace từng thay đổi liên kết và tự test các edge case (rỗng, toàn bộ bị xóa, xóa liên tiếp).

## 1. Đề bài

Cho `head` của một linked list và số nguyên `val`. Hãy **xóa mọi node** có `node.val == val` và trả về `head` mới.

| Input `head` | `val` | Output |
|---|:---:|---|
| `1 → 2 → 6 → 3 → 4 → 5 → 6` | 6 | `1 → 2 → 3 → 4 → 5` |
| rỗng (`None`) | 1 | rỗng |
| `7 → 7 → 7 → 7` | 7 | rỗng |

**Constraints:** số node trong `[0, 10^4]`; `1 <= Node.val <= 50`; `0 <= val <= 50`.

Điểm cần để ý ngay:
- `head` mới có thể **khác** `head` cũ (khi các node đầu bị xóa), thậm chí là `None`.
- Có thể phải xóa **nhiều node liền nhau** (ví dụ 3).

## 2. Nền tảng: `ListNode` — linked list không phải list Python

Một **node** có 2 trường:
- `val`: giá trị.
- `next`: tham chiếu (reference) tới node kế tiếp, hoặc `None` nếu là node cuối.

`head` là tham chiếu tới node đầu tiên. Khác list Python: không có chỉ số `[i]`, muốn tới node thứ 3 phải đi `head.next.next`.

LeetCode tự định nghĩa `ListNode` khi chấm; trong notebook ta tự viết để chạy được.

In [1]:
from typing import Optional

class ListNode:
    def __init__(self, val: int = 0, next: "Optional[ListNode]" = None):
        self.val = val
        self.next = next

a = ListNode(1)
b_node = ListNode(2)
c = ListNode(6)
a.next = b_node      # 1 -> 2
b_node.next = c      # 2 -> 6
print("a.val =", a.val, "| a.next.val =", a.next.val, "| a.next.next.val =", a.next.next.val)
print("c.next is None?", c.next is None)

a.val = 1 | a.next.val = 2 | a.next.next.val = 6
c.next is None? True


### 2.1. Hàm phụ dựng / in linked list (chỉ để học và test)

- `build_list([1, 2, 6])` tạo linked list từ list Python.
- `to_list(head)` đi theo `next` để thu lại list giá trị.
- `show(head)` in dạng mũi tên cho dễ nhìn.

`reversed(values)` duyệt list từ cuối lên đầu: ta dựng từ node cuối, mỗi node mới trỏ tới `head` cũ.

In [2]:
def build_list(values: list[int]) -> Optional[ListNode]:
    head = None
    for v in reversed(values):
        head = ListNode(v, head)
    return head

def to_list(head: Optional[ListNode]) -> list[int]:
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out

def show(head: Optional[ListNode]) -> str:
    return " -> ".join(map(str, to_list(head))) + " -> None" if head else "None"

h = build_list([1, 2, 6, 3, 4, 5, 6])
print(show(h))
print(to_list(h))
print(show(build_list([])))

1 -> 2 -> 6 -> 3 -> 4 -> 5 -> 6 -> None
[1, 2, 6, 3, 4, 5, 6]
None


### 2.2. `is None`, `is not None` và duyệt bằng `while`

- `node is None`: kiểm tra "không có node". Dùng `is` (so sánh danh tính) thay vì `==` cho `None` — đó là quy ước Python.
- Duyệt toàn bộ: `cur = head; while cur is not None: ...; cur = cur.next`.
- **Gán biến không sao chép node:** `cur = head` chỉ tạo thêm một "tên" cùng trỏ tới node đầu. Sửa `cur.val` cũng là sửa `head.val`.

In [3]:
head = build_list([5, 8, 9])
cur = head
while cur is not None:
    print("thăm node", cur.val)
    cur = cur.next
print("cur sau vòng lặp:", cur)

alias = head
alias.val = 99
print("Sửa qua alias -> head:", show(head))
print("alias is head?", alias is head)

thăm node 5
thăm node 8
thăm node 9
cur sau vòng lặp: None
Sửa qua alias -> head: 99 -> 8 -> 9 -> None
alias is head? True


## 3. "Xóa" node trong linked list nghĩa là gì?

Không có lệnh "xóa ô thứ i". Ta **đổi mũi tên** của node đứng trước để nhảy qua node cần xóa:

```text
trước:  prev -> X -> Y
lệnh:   prev.next = prev.next.next     (tức prev.next = Y)
sau:    prev -> Y        (X không còn ai trỏ tới, Python tự thu hồi bộ nhớ)
```

⇒ Muốn xóa `X`, ta phải đứng ở **node trước** `X`. Đây là chìa khóa của cả bài.

In [4]:
head = build_list([1, 2, 6, 3])
prev = head.next              # node 2
print("Trước:", show(head), "| prev =", prev.val, "| prev.next =", prev.next.val)
prev.next = prev.next.next    # bỏ qua node 6
print("Sau  :", show(head))

Trước: 1 -> 2 -> 6 -> 3 -> None | prev = 2 | prev.next = 6
Sau  : 1 -> 2 -> 3 -> None


## 4. Cách 1 — Baseline: tạo linked list mới chỉ chứa node giữ lại

Ý nghĩ đầu tiên: "duyệt từng node, node nào khác `val` thì **chép** sang danh sách mới".

Để nối node vào cuối danh sách mới, ta cần một **node giả** `dummy` đứng đầu và con trỏ `tail` luôn chỉ vào node cuối. Kết quả là `dummy.next`.

Nhược điểm: tạo thêm tối đa `n` node mới → bộ nhớ `O(n)`.

In [5]:
def remove_copy(head: Optional[ListNode], val: int) -> Optional[ListNode]:
    dummy = ListNode(0)
    tail = dummy
    cur = head
    while cur is not None:
        if cur.val != val:
            tail.next = ListNode(cur.val)   # tạo node MỚI
            tail = tail.next
        cur = cur.next
    return dummy.next

print(show(remove_copy(build_list([1, 2, 6, 3, 4, 5, 6]), 6)))
print(show(remove_copy(build_list([7, 7, 7, 7]), 7)))
print(show(remove_copy(None, 1)))

1 -> 2 -> 3 -> 4 -> 5 -> None
None
None


## 5. Cách 2 — Sửa tại chỗ, xử lý head riêng

Muốn bộ nhớ `O(1)`: không tạo node mới, chỉ đổi `next`.

**Vấn đề:** node đầu không có node đứng trước → không thể dùng `prev.next = ...`.

Giải pháp tự nhiên:
1. Trong khi `head` bị xóa (`head.val == val`), dời `head = head.next`. Dùng `while`, không phải `if`, vì có thể nhiều node đầu liền nhau cùng bị xóa (ví dụ `7 → 7 → 7`).
2. Sau đó `head` chắc chắn được giữ (hoặc là `None`). Dùng `prev = head`, xét `prev.next`:
   - nếu `prev.next.val == val` → xóa: `prev.next = prev.next.next` (**không** tiến `prev`, vì node mới nối vào cũng có thể cần xóa);
   - ngược lại → tiến `prev = prev.next`.

In [6]:
def remove_head_special(head: Optional[ListNode], val: int) -> Optional[ListNode]:
    while head is not None and head.val == val:   # bước 1: bỏ các node đầu
        head = head.next
    if head is None:
        return None
    prev = head
    while prev.next is not None:                   # bước 2: xét node phía sau prev
        if prev.next.val == val:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return head

print(show(remove_head_special(build_list([1, 2, 6, 3, 4, 5, 6]), 6)))
print(show(remove_head_special(build_list([7, 7, 7, 7]), 7)))
print(show(remove_head_special(build_list([6, 6, 1, 6, 6, 2]), 6)))

1 -> 2 -> 3 -> 4 -> 5 -> None
None
1 -> 2 -> None


### 5.1. `and` ngắn mạch (short-circuit) bảo vệ khỏi `None`

`head is not None and head.val == val`: nếu vế trái `False` thì Python **không tính** vế phải. Nhờ vậy khi `head` là `None`, ta không bị lỗi `AttributeError: 'NoneType' object has no attribute 'val'`.

Đảo thứ tự hai vế sẽ gây lỗi:

In [7]:
head = None
print("Đúng thứ tự:", head is not None and head.val == 6)
try:
    print(head.val == 6 and head is not None)
except AttributeError as e:
    print("Sai thứ tự -> lỗi:", e)

Đúng thứ tự: False
Sai thứ tự -> lỗi: 'NoneType' object has no attribute 'val'


## 6. Nút thắt & insight — vì sao cần dummy node?

Cách 2 có **hai đoạn code** gần giống nhau: một đoạn cho head, một đoạn cho phần còn lại. Code trùng lặp dễ sai (quên `while`, quên `return None`...).

**Insight:** nếu head cũng **có node đứng trước** thì mọi node được xử lý giống nhau. Ta tự tạo node đó:

```python
dummy = ListNode(0, head)     # dummy -> head -> ...
```

Bây giờ "xóa head" chỉ là `dummy.next = dummy.next.next`. Cuối cùng trả `dummy.next` — đó chính là head mới (có thể là `None`).

Giá trị của `dummy` không quan trọng (không bao giờ được so sánh), ta để 0.

## 7. Cách 3 — Lời giải chính: dummy node + con trỏ `prev`

Từng đoạn:

**Đoạn 1 — tạo node giả đứng trước head**
```python
dummy = ListNode(0, head)
prev = dummy
```

**Đoạn 2 — luôn nhìn node *phía sau* `prev`**
```python
while prev.next is not None:
```

**Đoạn 3 — quyết định xóa hay đi tiếp**
```python
    if prev.next.val == val:
        prev.next = prev.next.next   # xóa, prev đứng yên
    else:
        prev = prev.next             # giữ, prev tiến lên
```

**Đoạn 4 — trả head mới**
```python
return dummy.next
```

In [8]:
def remove_dummy(head: Optional[ListNode], val: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next is not None:
        if prev.next.val == val:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return dummy.next

for vals, v in [([1, 2, 6, 3, 4, 5, 6], 6), ([], 1), ([7, 7, 7, 7], 7), ([1, 6, 6, 2], 6)]:
    print(f"{vals!s:<24} val={v} -> {show(remove_dummy(build_list(vals), v))}")

[1, 2, 6, 3, 4, 5, 6]    val=6 -> 1 -> 2 -> 3 -> 4 -> 5 -> None
[]                       val=1 -> None
[7, 7, 7, 7]             val=7 -> None
[1, 6, 6, 2]             val=6 -> 1 -> 2 -> None


## 8. TRACE từng bước

### 8.1. Bảng trace `1 → 6 → 6 → 2`, `val = 6` (có hai node cần xóa liền nhau)

`D` là dummy. Cột `prev.next` là node đang được xét.

| Bước | `prev` | `prev.next` | Bằng 6? | Hành động | Danh sách sau bước |
|---:|---|---|:---:|---|---|
| 1 | D | 1 | không | `prev = 1` | D → 1 → 6 → 6 → 2 |
| 2 | 1 | 6 | **có** | `1.next = 6(thứ 2)` | D → 1 → 6 → 2 |
| 3 | 1 | 6 | **có** | `1.next = 2` | D → 1 → 2 |
| 4 | 1 | 2 | không | `prev = 2` | D → 1 → 2 |
| 5 | 2 | None | — | dừng | trả `dummy.next` = 1 → 2 |

Bước 2–3 cho thấy vì sao **không tiến `prev` sau khi xóa**: node mới nối vào (6 thứ hai) cũng cần xóa.

In [9]:
def trace_remove(head: Optional[ListNode], val: int) -> Optional[ListNode]:
    dummy = ListNode("D", head)
    prev = dummy
    step = 1
    while prev.next is not None:
        target = prev.next
        if target.val == val:
            prev.next = target.next
            action = f"XÓA node {target.val}, prev đứng yên ở {prev.val}"
        else:
            prev = target
            action = f"giữ node {target.val}, prev tiến tới {prev.val}"
        print(f"Bước {step}: xét {target.val} -> {action:<34} | {show(dummy.next)}")
        step += 1
    print("Kết quả:", show(dummy.next))
    return dummy.next

trace_remove(build_list([1, 6, 6, 2]), 6)

Bước 1: xét 1 -> giữ node 1, prev tiến tới 1        | 1 -> 6 -> 6 -> 2 -> None
Bước 2: xét 6 -> XÓA node 6, prev đứng yên ở 1      | 1 -> 6 -> 2 -> None
Bước 3: xét 6 -> XÓA node 6, prev đứng yên ở 1      | 1 -> 2 -> None
Bước 4: xét 2 -> giữ node 2, prev tiến tới 2        | 1 -> 2 -> None
Kết quả: 1 -> 2 -> None


### 8.2. Trace trường hợp head bị xóa liên tục — `7 → 7 → 7 → 7`, `val = 7`

Dummy giúp ta không cần nhánh đặc biệt: `prev` đứng yên ở `D` suốt, mỗi bước cắt một node 7.

In [10]:
trace_remove(build_list([7, 7, 7, 7]), 7)

Bước 1: xét 7 -> XÓA node 7, prev đứng yên ở D      | 7 -> 7 -> 7 -> None
Bước 2: xét 7 -> XÓA node 7, prev đứng yên ở D      | 7 -> 7 -> None
Bước 3: xét 7 -> XÓA node 7, prev đứng yên ở D      | 7 -> None
Bước 4: xét 7 -> XÓA node 7, prev đứng yên ở D      | None
Kết quả: None


In [11]:
trace_remove(build_list([1, 2, 6, 3, 4, 5, 6]), 6)

Bước 1: xét 1 -> giữ node 1, prev tiến tới 1        | 1 -> 2 -> 6 -> 3 -> 4 -> 5 -> 6 -> None
Bước 2: xét 2 -> giữ node 2, prev tiến tới 2        | 1 -> 2 -> 6 -> 3 -> 4 -> 5 -> 6 -> None
Bước 3: xét 6 -> XÓA node 6, prev đứng yên ở 2      | 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> None
Bước 4: xét 3 -> giữ node 3, prev tiến tới 3        | 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> None
Bước 5: xét 4 -> giữ node 4, prev tiến tới 4        | 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> None
Bước 6: xét 5 -> giữ node 5, prev tiến tới 5        | 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> None
Bước 7: xét 6 -> XÓA node 6, prev đứng yên ở 5      | 1 -> 2 -> 3 -> 4 -> 5 -> None
Kết quả: 1 -> 2 -> 3 -> 4 -> 5 -> None


## 9. Cách 4 — Đệ quy (recursion)

Ý tưởng "chia nhỏ": danh sách = `head` + **phần còn lại**.
1. **Trường hợp cơ sở:** `head is None` → trả `None`.
2. Xóa đệ quy trên phần còn lại: `head.next = remove(head.next, val)`.
3. Nếu `head.val == val` thì bỏ `head`, trả `head.next`; ngược lại trả `head`.

Đọc như tiếng Việt: "dọn sạch phần sau trước, rồi quyết định có giữ mình không".

Lưu ý: mỗi lời gọi đệ quy chiếm một **khung trên call stack** → bộ nhớ `O(n)`. Python mặc định giới hạn độ sâu đệ quy khoảng 1000, nên với 10^4 node bản đệ quy có thể lỗi `RecursionError` (LeetCode thường nâng giới hạn này, nhưng không nên dựa vào).

In [12]:
def remove_recursive(head: Optional[ListNode], val: int) -> Optional[ListNode]:
    if head is None:
        return None
    head.next = remove_recursive(head.next, val)
    return head.next if head.val == val else head

print(show(remove_recursive(build_list([1, 2, 6, 3, 4, 5, 6]), 6)))
print(show(remove_recursive(build_list([7, 7, 7, 7]), 7)))

import sys
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())

1 -> 2 -> 3 -> 4 -> 5 -> None
None
Giới hạn đệ quy mặc định: 1000


In [13]:
def trace_recursive(head, val, depth=0):
    pad = "  " * depth
    if head is None:
        print(pad + "gọi(None) -> trả None")
        return None
    print(pad + f"gọi({head.val}) -> xử lý phần sau trước")
    head.next = trace_recursive(head.next, val, depth + 1)
    if head.val == val:
        print(pad + f"node {head.val} == {val} -> BỎ, trả {head.next.val if head.next else None}")
        return head.next
    print(pad + f"node {head.val} giữ lại -> trả {head.val}")
    return head

print("Kết quả:", show(trace_recursive(build_list([6, 1, 6]), 6)))

gọi(6) -> xử lý phần sau trước
  gọi(1) -> xử lý phần sau trước
    gọi(6) -> xử lý phần sau trước
      gọi(None) -> trả None
    node 6 == 6 -> BỎ, trả None
  node 1 giữ lại -> trả 1
node 6 == 6 -> BỎ, trả 1
Kết quả: 1 -> None


## 10. Code nộp LeetCode

Bản chính: dummy node (không cần định nghĩa lại `ListNode` khi nộp — LeetCode đã có sẵn).

In [14]:
class Solution:
    def removeElements(self, head: Optional[ListNode], val: int) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        prev = dummy
        while prev.next:
            if prev.next.val == val:
                prev.next = prev.next.next
            else:
                prev = prev.next
        return dummy.next

print(show(Solution().removeElements(build_list([1, 2, 6, 3, 4, 5, 6]), 6)))

1 -> 2 -> 3 -> 4 -> 5 -> None


Bản đệ quy (ngắn, đẹp, nhưng tốn stack):

```python
class Solution:
    def removeElements(self, head, val):
        if not head:
            return None
        head.next = self.removeElements(head.next, val)
        return head.next if head.val == val else head
```

## 11. Vì sao đúng? (invariant)

Với cách dummy, ở đầu mỗi vòng lặp:

> **Invariant:** các node từ `dummy.next` đến `prev` (nếu `prev != dummy`) đều có `val != val_cần_xóa`, và là đúng thứ tự ban đầu; phần từ `prev.next` trở đi là phần **chưa xét** của danh sách gốc.

- **Ban đầu:** `prev = dummy`, đoạn "đã xét" rỗng → đúng.
- **Nhánh xóa:** `prev.next` bị cắt, phần chưa xét mất đúng một node có giá trị cần xóa; đoạn đã xét không đổi → vẫn đúng.
- **Nhánh giữ:** `prev.next` có giá trị khác, ta đưa nó vào đoạn đã xét → vẫn đúng.
- **Kết thúc:** `prev.next is None` → không còn phần chưa xét → `dummy.next` là toàn bộ các node hợp lệ.
- **Dừng:** mỗi vòng hoặc cắt một node, hoặc tiến `prev` — phần chưa xét giảm đúng 1 node mỗi vòng.

## 12. Time / Space — so sánh 4 cách

Gọi `n` là số node.

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| 1. Tạo list mới | chép node giữ lại | `O(n)` | `O(n)` | không sửa list gốc |
| 2. Head riêng | `while` xóa head rồi `prev` | `O(n)` | `O(1)` | hai đoạn code giống nhau |
| 3. Dummy node | node giả trước head | `O(n)` | `O(1)` | **gọn, nên dùng** |
| 4. Đệ quy | xử lý phần sau trước | `O(n)` | `O(n)` call stack | có thể vượt giới hạn đệ quy |

Mỗi node được xét đúng một lần ở cả 4 cách.

## 13. Bộ test

1. Ba ví dụ đề.
2. Biên: 1 node bị xóa / không bị xóa; `val = 0` (không node nào có giá trị 0 vì `Node.val >= 1`).
3. Xóa ở đầu, giữa, cuối; xóa liền nhau; không xóa gì.
4. Danh sách dài `10^4` node (bản lặp).
5. Đối chiếu 4 cách với cách "lọc list Python" trên dữ liệu ngẫu nhiên có seed.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
run = lambda vals, v: to_list(sol.removeElements(build_list(vals), v))
check("ví dụ 1", run([1, 2, 6, 3, 4, 5, 6], 6), [1, 2, 3, 4, 5])
check("ví dụ 2 (rỗng)", run([], 1), [])
check("ví dụ 3 (xóa hết)", run([7, 7, 7, 7], 7), [])
check("1 node bị xóa", run([5], 5), [])
check("1 node giữ lại", run([5], 4), [5])
check("val=0 không xóa gì", run([1, 2, 3], 0), [1, 2, 3])
check("xóa đầu", run([3, 1, 2], 3), [1, 2])
check("xóa cuối", run([1, 2, 3], 3), [1, 2])
check("xóa xen kẽ", run([2, 1, 2, 1, 2], 2), [1, 1])
big = list(range(1, 51)) * 200            # 10000 node
check("10^4 node", len(run(big, 7)), 10000 - 200)

PASS ví dụ 1: [1, 2, 3, 4, 5]
PASS ví dụ 2 (rỗng): []
PASS ví dụ 3 (xóa hết): []
PASS 1 node bị xóa: []
PASS 1 node giữ lại: [5]
PASS val=0 không xóa gì: [1, 2, 3]
PASS xóa đầu: [1, 2]
PASS xóa cuối: [1, 2]
PASS xóa xen kẽ: [1, 1]
PASS 10^4 node: 9800


In [16]:
import random
rng = random.Random(203)
methods = [remove_copy, remove_head_special, remove_dummy, remove_recursive,
           Solution().removeElements]
for _ in range(3000):
    vals = [rng.randint(1, 4) for _ in range(rng.randint(0, 12))]
    v = rng.randint(0, 4)
    expected = [x for x in vals if x != v]          # oracle: lọc list Python
    for m in methods:
        assert to_list(m(build_list(vals), v)) == expected, (m.__name__, vals, v)
print("5 cài đặt khớp oracle trên 3000 input ngẫu nhiên!")

5 cài đặt khớp oracle trên 3000 input ngẫu nhiên!


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Tiến `prev` cả khi vừa xóa

```python
while prev.next:
    if prev.next.val == val:
        prev.next = prev.next.next
    prev = prev.next          # SAI: luôn tiến
```
Hậu quả: bỏ sót node cần xóa đứng ngay sau node vừa xóa, và có thể `prev` thành `None` → lỗi.

In [17]:
def remove_bug_always_advance(head, val):
    dummy = ListNode(0, head)
    prev = dummy
    while prev is not None and prev.next is not None:
        if prev.next.val == val:
            prev.next = prev.next.next
        prev = prev.next
    return dummy.next

print("Bản lỗi  [1,6,6,2], 6 ->", to_list(remove_bug_always_advance(build_list([1, 6, 6, 2]), 6)))
print("Bản đúng [1,6,6,2], 6 ->", to_list(remove_dummy(build_list([1, 6, 6, 2]), 6)))

Bản lỗi  [1,6,6,2], 6 -> [1, 6, 2]
Bản đúng [1,6,6,2], 6 -> [1, 2]


### Lỗi 2 — Dùng `if` thay `while` khi xử lý head riêng

```python
if head and head.val == val:   # SAI: chỉ bỏ được MỘT node đầu
    head = head.next
```
Với `7 → 7 → 7 → 7`: `if` chỉ bỏ node 7 thứ nhất, `head` mới vẫn là một node 7; vòng `prev` chỉ xét các node **sau** `head` nên kết quả còn `[7]` thay vì `[]`.

In [18]:
def remove_bug_if_head(head, val):
    if head is not None and head.val == val:
        head = head.next
    if head is None:
        return None
    prev = head
    while prev.next:
        if prev.next.val == val:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return head

print("Bản lỗi [7,7,7,7], 7 ->", to_list(remove_bug_if_head(build_list([7, 7, 7, 7]), 7)), "(đúng phải là [])")

Bản lỗi [7,7,7,7], 7 -> [7] (đúng phải là [])


### Lỗi 3 — Trả `head` thay vì `dummy.next`

Nếu node đầu bị xóa, biến `head` vẫn trỏ vào node cũ (đã bị cắt khỏi chuỗi). Trả `head` sẽ trả về node lẽ ra đã xóa.

In [19]:
def remove_bug_return_head(head, val):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        if prev.next.val == val:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return head                     # SAI

print("Bản lỗi [6,1,2], 6 ->", to_list(remove_bug_return_head(build_list([6, 1, 2]), 6)), "(đúng phải là [1, 2])")

Bản lỗi [6,1,2], 6 -> [6, 1, 2] (đúng phải là [1, 2])


### Lỗi 4 — Truy cập `.val` của `None`

Viết `while cur.next.val ...` hoặc quên kiểm tra rỗng → `AttributeError: 'NoneType' object has no attribute 'val'`. Luôn kiểm tra `is not None` trước khi đọc `.val` / `.next`.

### Lỗi 5 — Nhầm "xóa node" với "xóa giá trị khỏi list Python"

`to_list(head).remove(6)` chỉ xóa trên một list Python **tạm** (và chỉ xóa lần xuất hiện đầu); linked list gốc không đổi. Đề yêu cầu trả về **ListNode**.

## 15. Mẹo nhận dạng pattern

- Đề có thể làm **thay đổi head** (xóa đầu, chèn đầu, đảo, gộp) → nghĩ ngay tới **dummy node**.
- Muốn xóa node `X` → phải đứng ở **node trước** `X` và nhìn `prev.next`.
- Sau khi xóa, **đứng yên**; chỉ tiến khi giữ node.
- Bài cùng khuôn: 83 Remove Duplicates from Sorted List, 82 Remove Duplicates II, 19 Remove Nth Node From End, 21 Merge Two Sorted Lists (dummy + tail), 237 Delete Node in a Linked List (biến thể không có prev).

## 16. Biến thể

### 16.1. Xóa theo điều kiện bất kỳ (`predicate`)
Thay `prev.next.val == val` bằng `should_remove(prev.next.val)`. Ví dụ xóa mọi số chẵn.

### 16.2. Trả về cả số node đã xóa
Thêm bộ đếm trong nhánh xóa.

### 16.3. LeetCode 237 — chỉ được cho node cần xóa, không có head
Không có `prev`! Mẹo: chép giá trị node sau sang node hiện tại rồi xóa node sau: `node.val = node.next.val; node.next = node.next.next`.

In [20]:
def remove_if(head, should_remove):
    dummy = ListNode(0, head)
    prev = dummy
    removed = 0
    while prev.next:
        if should_remove(prev.next.val):
            prev.next = prev.next.next
            removed += 1
        else:
            prev = prev.next
    return dummy.next, removed

new_head, cnt = remove_if(build_list([1, 2, 3, 4, 5, 6]), lambda x: x % 2 == 0)
print("Xóa số chẵn:", show(new_head), "| đã xóa:", cnt)

def delete_node(node):                 # LeetCode 237
    node.val = node.next.val
    node.next = node.next.next

h = build_list([4, 5, 1, 9])
delete_node(h.next)                    # xóa node 5
print("237 xóa node 5:", show(h))

Xóa số chẵn: 1 -> 3 -> 5 -> None | đã xóa: 3
237 xóa node 5: 4 -> 1 -> 9 -> None


`lambda x: x % 2 == 0` là **hàm vô danh** viết trên một dòng: nhận `x`, trả `x % 2 == 0`. Tương đương:

```python
def is_even(x):
    return x % 2 == 0
```

## 17. Bài tập tự luyện

### Bài 1 — Vẽ tay
Vẽ danh sách và vị trí `prev` sau mỗi bước khi chạy cách dummy trên `2 → 2 → 1 → 2`, `val = 2`.

### Bài 2 — Dự đoán
Với cách dummy, vòng `while` chạy bao nhiêu lần trên danh sách `n` node? Có phụ thuộc số node bị xóa không?

### Bài 3 — TODO: tự viết cách dummy

In [21]:
def exercise_remove(head, val):
    # TODO 1: tạo dummy trỏ tới head, prev = dummy
    # TODO 2: khi prev.next còn: xóa (đứng yên) hoặc tiến
    # TODO 3: trả dummy.next
    pass

# assert to_list(exercise_remove(build_list([1, 2, 6, 3, 4, 5, 6]), 6)) == [1, 2, 3, 4, 5]
# assert to_list(exercise_remove(build_list([7, 7, 7]), 7)) == []

### Bài 4 — TODO: tự viết bản đệ quy

In [22]:
def exercise_remove_recursive(head, val):
    # TODO: trường hợp cơ sở + xử lý phần sau trước
    pass

# assert to_list(exercise_remove_recursive(build_list([6, 1, 6]), 6)) == [1]

### Bài 5 — Biến thể: xóa các node có giá trị nằm trong một tập `bad`
Ví dụ `bad = {2, 4}`: `1 → 2 → 3 → 4 → 5` → `1 → 3 → 5`.

In [23]:
def exercise_remove_set(head, bad: set[int]):
    # TODO: giống dummy, đổi điều kiện xóa
    pass

# assert to_list(exercise_remove_set(build_list([1, 2, 3, 4, 5]), {2, 4})) == [1, 3, 5]

### Bài 6 — Mini challenge: chỉ giữ lần xuất hiện **đầu tiên** của mỗi giá trị (danh sách không sắp xếp)
Ví dụ `3 → 1 → 3 → 2 → 1` → `3 → 1 → 2`. Gợi ý: kết hợp dummy với một `set` "đã thấy".

In [24]:
def exercise_keep_first(head):
    # TODO: dummy + set seen
    pass

# assert to_list(exercise_keep_first(build_list([3, 1, 3, 2, 1]))) == [3, 1, 2]

## Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | `prev` đứng yên khi xóa | bắt đầu `prev = D` | D xóa 2, xóa 2, tiến tới 1, xóa 2 |
| 2 | Mỗi vòng làm gì với phần chưa xét? | mỗi vòng bớt 1 node | đúng `n` lần |
| 3 | Node giả trước head | `ListNode(0, head)` | `if prev.next.val == val: prev.next = prev.next.next else: prev = prev.next` |
| 4 | Dọn phần sau trước | `head.next = f(head.next, val)` | `return head.next if head.val == val else head` |
| 5 | Chỉ đổi điều kiện | `x in bad` | `if prev.next.val in bad:` |
| 6 | Đã thấy → xóa | `seen = set()` | nhánh giữ: `seen.add(prev.next.val)` |

## Đáp án tham khảo

**Bài 1:** `D→2→2→1→2`: xét 2 → xóa (`D→2→1→2`, prev=D) → xét 2 → xóa (`D→1→2`, prev=D) → xét 1 → giữ (prev=1) → xét 2 → xóa (`D→1`) → dừng. Kết quả `1`.

**Bài 2:** Đúng `n` lần, **không** phụ thuộc số node bị xóa — mỗi vòng phần chưa xét giảm đúng 1 node (hoặc bị cắt, hoặc được giữ).

In [25]:
def answer_remove(head, val):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        if prev.next.val == val:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return dummy.next

def answer_remove_recursive(head, val):
    if head is None:
        return None
    head.next = answer_remove_recursive(head.next, val)
    return head.next if head.val == val else head

def answer_remove_set(head, bad):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        if prev.next.val in bad:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return dummy.next

def answer_keep_first(head):
    dummy = ListNode(0, head)
    prev = dummy
    seen = set()
    while prev.next:
        if prev.next.val in seen:
            prev.next = prev.next.next
        else:
            seen.add(prev.next.val)
            prev = prev.next
    return dummy.next

def count_loops(head, val):
    dummy = ListNode(0, head); prev = dummy; loops = 0
    while prev.next:
        loops += 1
        if prev.next.val == val: prev.next = prev.next.next
        else: prev = prev.next
    return loops

assert to_list(answer_remove(build_list([2, 2, 1, 2]), 2)) == [1]
assert count_loops(build_list([2, 2, 1, 2]), 2) == 4 and count_loops(build_list([1, 3, 5]), 2) == 3
assert to_list(answer_remove(build_list([1, 2, 6, 3, 4, 5, 6]), 6)) == [1, 2, 3, 4, 5]
assert to_list(answer_remove(build_list([7, 7, 7]), 7)) == []
assert to_list(answer_remove_recursive(build_list([6, 1, 6]), 6)) == [1]
assert to_list(answer_remove_set(build_list([1, 2, 3, 4, 5]), {2, 4})) == [1, 3, 5]
assert to_list(answer_keep_first(build_list([3, 1, 3, 2, 1]))) == [3, 1, 2]
print("Đáp án tham khảo PASS toàn bộ!")

Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
dummy = ListNode(0, head)      # node giả đứng trước head
prev = dummy
while prev.next:
    if CAN_XOA(prev.next):
        prev.next = prev.next.next   # xóa -> prev ĐỨNG YÊN
    else:
        prev = prev.next             # giữ -> prev TIẾN
return dummy.next              # head mới
```

**Checklist trước khi nộp:**
- [ ] Trả `dummy.next`, không trả `head` cũ.
- [ ] Chỉ tiến `prev` ở nhánh **giữ**.
- [ ] Đã thử danh sách rỗng, xóa toàn bộ, xóa node đầu, xóa liền nhau, xóa node cuối.
- [ ] Không đọc `.val` của `None`.
- [ ] Bản đệ quy: lưu ý độ sâu tối đa `10^4`.